# Tạo giao dịch Fashion-MNIST bằng túi từ thị giác nhị phân

Notebook này gọi pipeline `FashionTC` hiện có, theo thứ tự từng bước. Mỗi cell code chỉ thực hiện một giai đoạn để người đọc thấy rõ artifact trung gian và lỗi thuộc giai đoạn nào. Julia chạy toàn bộ quy trình; không có Python điều phối.

**Chạy notebook:** chọn kernel Julia CPU trên Colab. Cần mạng ở lần tải dữ liệu/source đầu tiên. Có thể chạy trong checkout local đúng commit được ghim.

**Mục lục**

1. Xác minh dữ liệu và chuẩn bị cấu hình
2. Tách patch ảnh
3. Lọc nền
4. Tạo descriptor
5. Học từ điển bằng K-means
6. Gán từ thị giác
7. Mã hóa giao dịch nhị phân
8. Kiểm định giao dịch
9. Truy vết ảnh và xuất gói bàn giao

Cấu hình giữ 70.000 ảnh, 32 cụm, seed 123 và dung sai 0,0001. Demo đặt giới hạn tối đa 400 vòng để bao trùm lượt Colab từng dừng ở vòng 372; đây là trần lặp, không ép số vòng. Kết quả từng lượt luôn được đọc từ artifact.

### Thiết lập Julia và source

Cell đầu tìm checkout đúng phiên bản; nếu không thấy, Colab tải commit đã ghim. Tiếp theo nạp project Julia và các hàm hiển thị dùng trong notebook.


In [ ]:
# Julia runtime: chạy trực tiếp bằng kernel Julia, không cần Python điều phối.
# Colab: chọn Runtime → Change runtime type → Julia và CPU trước khi chạy.
# JULIA_NUM_THREADS phải được đặt trước khi kernel khởi động; cell này chỉ ghi lại yêu cầu.
ENV["JULIA_NUM_THREADS"] = "1"
ENV["OPENBLAS_NUM_THREADS"] = "1"
ENV["OPENBLAS_CORETYPE"] = "Haswell"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

using Pkg, SHA, Markdown, LibGit2, Base64
using LinearAlgebra

REPOSITORY_URL = "https://github.com/NguyenVoTanDuy2006/DataMining-FIM-Project.git"
SOURCE_BRANCH = "Fashion-Mnist"
SOURCE_COMMIT = "7737fdef756d9bf77bc1cb7988f955de2f572012"

function find_fashion_repo(start_path)
    path = realpath(start_path)
    while true
        if isfile(joinpath(path, "Project.toml")) && isfile(joinpath(path, "src", "EclatFIM.jl"))
            return path
        end
        parent = dirname(path)
        parent == path && return nothing
        path = parent
    end
end

local_candidate = find_fashion_repo(pwd())
local_commit = if isnothing(local_candidate)
    nothing
else
    repo = LibGit2.GitRepo(local_candidate)
    try
        string(LibGit2.head_oid(repo))
    finally
        close(repo)
    end
end
USING_LOCAL_REPO = local_commit == SOURCE_COMMIT
if USING_LOCAL_REPO
    REPO_ROOT = local_candidate
    actual_commit = local_commit
else
    REPO_ROOT = joinpath("/content", "fashion-fim-" * first(SOURCE_COMMIT, 8))
    git_repo = if !ispath(REPO_ROOT)
        mkpath(dirname(REPO_ROOT))
        LibGit2.clone(REPOSITORY_URL, REPO_ROOT; branch=SOURCE_BRANCH)
    elseif isdir(joinpath(REPO_ROOT, ".git"))
        LibGit2.GitRepo(REPO_ROOT)
    else
        error("$REPO_ROOT exists but is not the pinned Git repository. Restart the Colab Julia runtime.")
    end
    actual_commit = try
        LibGit2.checkout!(git_repo, SOURCE_COMMIT; force=true)
        string(LibGit2.head_oid(git_repo))
    finally
        close(git_repo)
    end
end
actual_commit == SOURCE_COMMIT || error("Expected source $SOURCE_COMMIT, found $actual_commit. Restart the runtime.")

if USING_LOCAL_REPO
    DEPOT_ROOT = first(DEPOT_PATH)
    DEMO_OUTPUT_ROOT = joinpath(tempdir(), "fashion-demo-output-" * first(SOURCE_COMMIT, 8))
    mkpath(DEMO_OUTPUT_ROOT)
else
    DEPOT_ROOT = joinpath("/content", "fashion-demo-work", "julia_depot")
    mkpath(DEPOT_ROOT)
    DEPOT_ROOT in DEPOT_PATH || pushfirst!(DEPOT_PATH, DEPOT_ROOT)
    DEMO_OUTPUT_ROOT = joinpath("/content", "fashion-demo-output")
    mkpath(DEMO_OUTPUT_ROOT)
end
ENV["JULIA_DEPOT_PATH"] = join(DEPOT_PATH, ":")
ENV["FASHION_DEMO_REPO_ROOT"] = REPO_ROOT
ENV["FASHION_DEMO_SOURCE_COMMIT"] = SOURCE_COMMIT
ENV["FASHION_DEMO_OUTPUT_ROOT"] = DEMO_OUTPUT_ROOT
ENV["FASHION_DEMO_LOCAL_REPO"] = USING_LOCAL_REPO ? "1" : "0"


#### Nạp project Julia và tiện ích hiển thị

Kích hoạt project đã ghim, nạp pipeline Fashion-MNIST, giới hạn BLAS ở một luồng và chuẩn bị các hàm hiển thị dùng ở những phần sau.

In [ ]:
Pkg.activate(REPO_ROOT)
Pkg.instantiate()
using EclatFIM
using EclatFIM.FashionMNIST: FashionTC, FashionFIMWorkflow
BLAS.set_num_threads(1)

function show_demo_markdown(path)
    display(Markdown.parse(read(path, String)))
end
function show_demo_image(path; mime="image/png", alt=basename(path))
    encoded = base64encode(read(path))
    safe_alt = replace(alt, "&" => "&amp;", "\"" => "&quot;", "<" => "&lt;", ">" => "&gt;")
    html = "<figure><img alt=\"$safe_alt\" src=\"data:$mime;base64,$encoded\" style=\"max-width:100%;height:auto\"><figcaption>$safe_alt</figcaption></figure>"
    display(MIME"text/html"(), html)
end

println("Source commit: ", actual_commit)
println("Execution mode: ", USING_LOCAL_REPO ? "local repository" : "Google Colab")
println("Julia runtime: ", VERSION)
println("Active Julia threads: ", Threads.nthreads(), " (fixed when the kernel started)")
println("OpenBLAS threads: ", BLAS.get_num_threads(), "; requested core target: ", ENV["OPENBLAS_CORETYPE"])
println("Julia depot: ", DEPOT_ROOT)
println("Demo outputs: ", DEMO_OUTPUT_ROOT)
println("EclatFIM and Fashion-MNIST are ready in this Julia kernel.")

## 1. Xác minh dữ liệu và chuẩn bị cấu hình

Tải hoặc kiểm checksum bốn tệp dữ liệu, ghi manifest đầu vào, rồi tạo trạng thái lượt chạy từ cấu hình phát hành. Cell này chưa chạy các giai đoạn TC.

In [ ]:
using EclatFIM.FashionMNIST: FashionTC, FashionFIMWorkflow, create_dataset_manifest,
    download_fashion_mnist, new_release_run_id, parse_image_id
using SHA, TOML, Tar, CodecZlib, PNGFiles

repo_root = ENV["FASHION_DEMO_REPO_ROOT"]
source_commit = ENV["FASHION_DEMO_SOURCE_COMMIT"]
data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
release_profile = joinpath(repo_root, "src", "applications", "Fashion-Mnist", "configs", "transactions", "release.toml")
output_root = joinpath(repo_root, "results", "fashion", "release")
max_iterations = 400
run_id = new_release_run_id()
run_root = joinpath(output_root, run_id)
println("Run ID: ", run_id)
println("Checking the four official Fashion-MNIST archives...")
download_fashion_mnist(data_dir)
dataset = create_dataset_manifest(data_dir, run_root; repo_root, run_id, download_missing=false)
println("Dataset manifest: ", dataset.manifest_path)
println("Verified images: train=", length(dataset.train.labels), ", test=", length(dataset.test.labels))

profile = TOML.parsefile(release_profile)
profile["run"]["run_id"] = run_id
profile["tc4_codebook"]["max_iterations"] = max_iterations
profile["execution"]["backend"] == "cpu" || error("The demo profile must use CPU.")
config_dir = mktempdir(prefix="fashion-demo-config-")
config_path = joinpath(config_dir, "release.toml")
open(config_path, "w") do io
    TOML.print(io, profile)
end
state = FashionTC.prepare_run(config_path; repo_root, data_dir, dataset_manifest_path=dataset.manifest_path)
rm(config_dir; recursive=true, force=true)
println("Prepared configuration: 70,000 images, K=32, seed=123, max_iterations=", max_iterations, ", tolerance=0.0001")

## 2. Tách patch ảnh

Mỗi ảnh 28×28 được chia thành 49 patch 4×4, không chồng lấp. Chạy TC1 để ghi các patch và chỉ mục vị trí.

In [ ]:
tc1_summary = FashionTC.run_tc1!(state)
println(tc1_summary)

## 3. Lọc nền

TC2 giữ patch có ít nhất một pixel lớn hơn 29. Pixel đúng 29 không đạt điều kiện nghiêm ngặt.

In [ ]:
tc2_summary = FashionTC.run_tc2!(state)
println(tc2_summary)

## 4. Tạo descriptor

TC3 đổi 16 pixel của mỗi patch được giữ thành descriptor Float32 trong [0, 1]. Nhãn và tọa độ được lưu ngoài descriptor.


In [ ]:
tc3_summary = FashionTC.run_tc3!(state)
println(tc3_summary)

## 5. Học từ điển bằng K-means

TC4 fit K=32 trên descriptor đã giữ. Lượt chạy giới hạn Julia và OpenBLAS ở một luồng. Số vòng, cờ hội tụ và lý do dừng được đọc sau khi TC4 ghi metrics.

In [ ]:
tc4_summary = FashionTC.run_tc4!(state)
println(tc4_summary)
tc_run_id = only(state.run_ids)
tc_run_dir = state.run_dirs[tc_run_id]
metrics_path = joinpath(tc_run_dir, "tc4_metrics.toml")
fit_metrics = TOML.parsefile(metrics_path)
println("Status: ", fit_metrics["status"])
println("Converged: ", fit_metrics["converged"])
println("Iterations: ", fit_metrics["iterations"], "/", fit_metrics["max_iterations"])
println("Stop reason: ", fit_metrics["stop_reason"])

## 6. Gán từ thị giác

TC5 gán mỗi descriptor vào tâm gần nhất. Mã từ thị giác bắt đầu từ 1; vị trí patch vẫn nằm trong artifact truy vết.

In [ ]:
tc5_summary = FashionTC.run_tc5!(state)
println(tc5_summary)

## 7. Mã hóa giao dịch nhị phân

TC6 gom các từ thị giác khác nhau của mỗi ảnh thành một giao dịch đã sắp thứ tự. Giao dịch rỗng vẫn được giữ; nhãn lớp không đi vào JSONL.

In [ ]:
tc6_summary = FashionTC.run_tc6!(state)
println(tc6_summary)

## 8. Kiểm định giao dịch

TC7 kiểm tra hợp đồng artifact và mẫu số hỗ trợ. Cell dừng nếu lượt không tạo đúng 70.000 giao dịch hoặc manifest không đạt PASS.

In [ ]:
result = FashionTC.run_tc7!(state)
println(result)
length(result.runs) == 1 || error("The release profile must produce exactly one codebook run.")
run_result = only(result.runs)
run_result.transactions == 70_000 || error("Expected exactly 70,000 image transactions.")
run_result.status == "COMPLETED" || error("The transaction validation step did not complete.")
tc_run_id = only(state.run_ids)
tc_run_dir = state.run_dirs[tc_run_id]
metrics_path = joinpath(tc_run_dir, "tc4_metrics.toml")
fit_metrics = TOML.parsefile(metrics_path)
transaction_manifest_path = run_result.manifest_path
transaction_manifest = TOML.parsefile(transaction_manifest_path)
transaction_manifest["status"] == "PASS" || error("Transaction artifact manifest did not pass validation.")
transaction_path = joinpath(tc_run_dir, "transactions.jsonl")
input = FashionFIMWorkflow.load_transactions(transaction_path;
    expected_n=70_000, expected_items=1:32, require_nonempty=false, unique_mapping=true)
input.tid == collect(1:70_000) || error("Transaction identifiers are not 1 through 70,000.")
input.input_sha256 == transaction_manifest["transaction_sha256"] || error("Transaction SHA-256 does not match the artifact manifest.")
println("Validation: PASS; transactions=", length(input.transactions), "; empty=", input.empty_count)
println("Transaction SHA-256: ", input.input_sha256)

## 9. Truy vết ảnh và vùng ảnh

Chọn bản ghi gán đầu tiên, nối nó với descriptor gốc và kiểm tra từ thị giác đó có trong giao dịch của ảnh.

In [ ]:
assignment_path = joinpath(tc_run_dir, "patch_assignments.csv")
trace = open(assignment_path, "r") do io
    readline(io) == "image_id,patch_id,patch_row,patch_col,word_id" || error("Unexpected patch assignment header.")
    fields = split(readline(io), ',')
    length(fields) == 5 || error("No retained patch assignment is available.")
    (; image_id=fields[1], patch_id=parse(Int, fields[2]), patch_row=parse(Int, fields[3]),
        patch_col=parse(Int, fields[4]), word_id=parse(Int, fields[5]))
end
split_name, source_index = parse_image_id(trace.image_id)
image_slot = split_name == :train ? source_index + 1 : 60_000 + source_index + 1
image_slot <= length(state.image_ids) || error("The traced image is outside the release dataset.")
global_image_id = state.image_ids[image_slot]
descriptor_position = findfirst(row -> row.image_id == global_image_id && row.patch_id == trace.patch_id, state.descriptor_index)
isnothing(descriptor_position) && error("The exported patch has no matching descriptor.")
transaction_slot = findfirst(==(trace.image_id), string.(input.image_id))
isnothing(transaction_slot) && error("The example image is missing from the transaction table.")
trace.word_id in input.transactions[transaction_slot] || error("The assigned word is absent from the image transaction.")
y0, x0 = (trace.patch_row - 1)*4 + 1, (trace.patch_col - 1)*4 + 1
source_image = state.images[:, :, image_slot]
source_patch = source_image[y0:y0+3, x0:x0+3]
sample_descriptor = state.descriptor_matrix[:, descriptor_position]
output_dir = joinpath(ENV["FASHION_DEMO_OUTPUT_ROOT"], "transaction-construction", run_id)
mkpath(output_dir)
PNGFiles.save(joinpath(output_dir, "trace_image.png"), repeat(source_image; inner=(8, 8)))
PNGFiles.save(joinpath(output_dir, "trace_patch.png"), repeat(source_patch; inner=(16, 16)))
println("Image: ", trace.image_id, "; TID: ", input.tid[transaction_slot], "; patch: ", trace.patch_id)
println("Visual word: ", trace.word_id, "; image transaction: [", join(input.transactions[transaction_slot], ", "), "]")
println("Descriptor (16 values): [", join(round.(sample_descriptor; digits=4), ", "), "]")

### Đóng gói artifact để chuyển sang notebook 03

Gói chứa manifest, metrics K-means và đủ tám artifact giao dịch. Trước khi đóng gói, notebook so checksum từng file với manifest.

In [ ]:
bundle_workspace = mktempdir(prefix="fashion-demo-bundle-")
bundle_stage = joinpath(bundle_workspace, "bundle")
mkpath(bundle_stage)
function add_bundle_file!(relative_path)
    source_path = joinpath(repo_root, split(relative_path, '/')...)
    isfile(source_path) || error("Required output is missing: $relative_path")
    destination = joinpath(bundle_stage, split(relative_path, '/')...)
    mkpath(dirname(destination))
    cp(source_path, destination; force=true)
    return bytes2hex(open(sha256, destination))
end

dataset_relative = relpath(dataset.manifest_path, repo_root)
transaction_manifest_relative = relpath(transaction_manifest_path, repo_root)
fit_metrics_relative = relpath(metrics_path, repo_root)
dataset_sha = add_bundle_file!(dataset_relative)
transaction_manifest_sha = add_bundle_file!(transaction_manifest_relative)
fit_metrics_sha = add_bundle_file!(fit_metrics_relative)
required_artifact_names = ("transactions", "codebook", "centroids", "patch_assignments",
    "tid_image_mapping", "image_metadata", "config", "integrity_report")
all(haskey(transaction_manifest["artifacts"], name) for name in required_artifact_names) ||
    error("The release manifest does not list all eight required artifacts.")
for artifact_name in required_artifact_names
    artifact = transaction_manifest["artifacts"][artifact_name]
    actual_sha = add_bundle_file!(artifact["path"])
    actual_sha == artifact["sha256"] || error("SHA-256 mismatch while collecting $(artifact["path"]).")
end

demo_manifest = Dict{String,Any}(
    "schema_version" => 1,
    "run_id" => run_id,
    "tc_run_id" => tc_run_id,
    "source_commit" => source_commit,
    "julia_version" => string(VERSION),
    "dataset_manifest" => Dict("path" => dataset_relative, "sha256" => dataset_sha),
    "transaction_manifest" => Dict("path" => transaction_manifest_relative, "sha256" => transaction_manifest_sha),
    "fit_metrics" => Dict("path" => fit_metrics_relative, "sha256" => fit_metrics_sha),
)
open(joinpath(bundle_stage, "demo_manifest.toml"), "w") do io
    TOML.print(io, demo_manifest)
end
mkpath(ENV["FASHION_DEMO_OUTPUT_ROOT"])
tar_path = joinpath(bundle_workspace, "fashion-demo-$(run_id).tar")
bundle_path = joinpath(ENV["FASHION_DEMO_OUTPUT_ROOT"], "fashion-demo-$(run_id).tar.gz")
Tar.create(bundle_stage, tar_path; portable=true)
open(bundle_path, "w") do io
    write(io, transcode(GzipCompressor, read(tar_path)))
end
bundle_sha = bytes2hex(open(sha256, bundle_path))
println("Bundle: ", bundle_path)
println("Bundle SHA-256: ", bundle_sha)
println("Transaction SHA-256: ", input.input_sha256)

### Kết quả truy vết và tâm từ thị giác

Lưới bên dưới hiển thị đủ 32 tâm cụm theo thứ tự mã từ 1 đến 32, đọc trái sang phải rồi từ trên xuống.

In [ ]:
centroid_path = joinpath(repo_root, split(transaction_manifest["artifacts"]["centroids"]["path"], '/')...)
centroids = zeros(Float32, 16, 32)
centroid_seen = falses(16, 32)
open(centroid_path, "r") do io
    readline(io) == "word_id,pixel_index,centroid_value" || error("Unexpected centroid table header.")
    for line in eachline(io)
        fields = split(line, ',')
        length(fields) == 3 || error("Malformed centroid row.")
        word_id, pixel_index = parse(Int, fields[1]), parse(Int, fields[2])
        1 <= word_id <= 32 && 1 <= pixel_index <= 16 || error("Centroid index outside the codebook.")
        centroid_seen[pixel_index, word_id] && error("Duplicate centroid coordinate.")
        centroid_seen[pixel_index, word_id] = true
        centroids[pixel_index, word_id] = parse(Float32, fields[3])
    end
end
all(centroid_seen) || error("Centroid table is incomplete.")
cell_size, gap = 16, 3
centroid_grid = fill(UInt8(255), 4*cell_size + 5*gap, 8*cell_size + 9*gap)
for word_id in 1:32
    row, column = div(word_id-1, 8), mod(word_id-1, 8)
    y0, x0 = gap + row*(cell_size+gap) + 1, gap + column*(cell_size+gap) + 1
    prototype = reshape(centroids[:, word_id], 4, 4)
    pixels = round.(UInt8, clamp.(prototype, 0, 1) .* 255)
    centroid_grid[y0:y0+cell_size-1, x0:x0+cell_size-1] .= repeat(pixels; inner=(4,4))
end
centroid_grid = repeat(centroid_grid; inner=(4, 4))
PNGFiles.save(joinpath(output_dir, "centroid_grid.png"), centroid_grid)

open(joinpath(output_dir, "construction_summary.md"), "w") do io
    println(io, "## Kết quả lượt dựng giao dịch\n")
    println(io, "- Mã lượt dữ liệu: ", run_id)
    println(io, "- Mã lượt từ điển: ", tc_run_id)
    println(io, "- Commit mã nguồn: ", source_commit)
    println(io, "- Phiên bản Julia: ", VERSION)
    println(io, "- Cấu hình: 70.000 ảnh, patch 4×4, mức xám > 29, descriptor 16 chiều, K=32, seed=123.")
    println(io, "- K-means: ", fit_metrics["status"], "; hội tụ: ", fit_metrics["converged"], "; số vòng: ", fit_metrics["iterations"], "/", fit_metrics["max_iterations"], ".")
    println(io, "- Kiểm định giao dịch: ", run_result.status, "; manifest: ", transaction_manifest["status"], ".")
    println(io, "- Giao dịch: ", length(input.transactions), "; giao dịch rỗng: ", input.empty_count, ".")
    println(io, "- SHA-256 giao dịch: ", input.input_sha256)
    println(io, "- Gói bàn giao: ", bundle_path)
    println(io, "- SHA-256 gói: ", bundle_sha)
    println(io, "\n## Ví dụ truy vết\n")
    println(io, "- Ảnh ", trace.image_id, ", TID ", input.tid[transaction_slot], ", patch ", trace.patch_id, ", từ thị giác ", trace.word_id, ".")
    println(io, "- Descriptor 16 chiều: [", join(round.(sample_descriptor; digits=4), ", "), "].")
end
println("Summary: ", joinpath(output_dir, "construction_summary.md"))
println("Bundle: ", bundle_path)
show_demo_markdown(joinpath(output_dir, "construction_summary.md"))
show_demo_image(joinpath(output_dir, "trace_image.png"); alt="Ảnh Fashion-MNIST được truy vết")
show_demo_image(joinpath(output_dir, "trace_patch.png"); alt="Patch được truy vết trong ảnh")
show_demo_image(joinpath(output_dir, "centroid_grid.png"); alt="32 tâm từ thị giác; thứ tự từ trái sang phải, từ trên xuống")
println("Trong Colab, tải gói bằng Files panel. Trong local, dùng trực tiếp đường dẫn gói ở trên.")